In [2]:
%pip install selenium webdriver-manager

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.5 MB 1.3 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/9.5 MB 1.4 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/9.5 MB 1.4 MB/s eta 0:00:07
   ------ --------------------------------- 1.6/9.5 MB 1.6 MB/s eta 0:00:05
   -------- ------------------------------- 2.1/9.5 MB 1.8 MB/s eta 0:00:05
   ---------- ----------------------------- 2.6/9.5 MB 2.0 MB/s eta 0:00:04
   -------------- ------------------------- 3.4/9.5 MB 2.2 MB/s eta 0:00:03
   ---------------- ----------------------- 3.9/9.5 MB 2.3 MB/s eta 0:00:03
   ------------------ --------------------- 4.5/9.5 MB 2.4 MB/s eta 0:00:03
   ----------------------- ---------------- 5.5/9.5 MB 2.5 MB/s eta 0:00:02
   -------------------------- ------------- 6.3/9.5 MB 2.7 MB/s eta 0:00:02
   -----------------------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# 1. 這裡直接放入網址
test_url = "https://www.agoda.com/zh-hk/grand-hotel/hotel/taipei-tw.html?ds=P44XehJcL%2BU8TKh2"

try:
    print("🚀 正在啟動 Chrome...")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    
    print(f"🌐 正在嘗試開啟網址: {test_url}")
    driver.get(test_url)
    
    time.sleep(5)
    print("✅ 成功開啟頁面！標題為:", driver.title)

except Exception as e:
    print("\n❌ 偵錯結果：")
    print(f"錯誤類型: {type(e).__name__}")
    print(f"完整錯誤訊息: {e}")

finally:
    if 'driver' in locals():
        driver.quit()



🚀 正在啟動 Chrome...
🌐 正在嘗試開啟網址: https://www.agoda.com/zh-hk/grand-hotel/hotel/taipei-tw.html?ds=P44XehJcL%2BU8TKh2
✅ 成功開啟頁面！標題為: 圓山大飯店 | 人氣優惠及套餐 - Agoda


In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 設定目標：你要抓取的飯店網址與數量
HOTEL_URL = "https://www.agoda.com/zh-hk/grand-hotel/hotel/taipei-tw.html"
MAX_REVIEWS = 15 

def run_agoda_spider(url, target_count):
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    all_results = []
    seen_ids = set()

    try:
        driver.get(url)
        wait = WebDriverWait(driver, 20)

        while len(all_results) < target_count:
            print(f"📜 正在處理頁面，目前已成功抓取 {len(all_results)} 則...")
            
            # 1. 深度滾動，確保評論區塊與分頁按鈕載入
            for _ in range(10):
                driver.execute_script("window.scrollBy(0, 800);")
                time.sleep(0.8)
            
            # 2. 定位所有評論卡片 (根據截圖中的 data-element-name)
            card_selector = '[data-element-name="review-comment"]'
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, card_selector)))
            cards = driver.find_elements(By.CSS_SELECTOR, card_selector)

            # 3. 解析當前頁面的卡片
            for card in cards:
                if len(all_results) >= target_count: break
                
                try:
                    # 排除飯店回覆：檢查是否有分數標籤
                    score_el = card.find_elements(By.CSS_SELECTOR, ".Review-comment-leftScore")
                    if not score_el: continue
                    
                    # 檢查 ID 防止重複抓取
                    r_id = card.get_attribute("data-review-id")
                    if not r_id or r_id in seen_ids: continue
                    seen_ids.add(r_id)

                    # 提取各欄位資料 (依據你的截圖定位)
                    score = score_el[0].text
                    name = card.find_element(By.CSS_SELECTOR, '[data-info-type="reviewer-name"]').text
                    stay_date = card.find_element(By.CSS_SELECTOR, '[data-info-type="stay-detail"]').text
                    content = card.find_element(By.CSS_SELECTOR, ".Review-comment-bodyText").text.replace('\n', ' ').strip()
                    
                    # 發表日期：搜尋包含「評價」或「202」字眼的 span
                    comment_date = "未知日期"
                    spans = card.find_elements(By.TAG_NAME, "span")
                    for s in spans:
                        if "202" in s.text and ("月" in s.text or "評價" in s.text):
                            comment_date = s.text
                            break

                    all_results.append({
                        "id": r_id, "name": name, "score": score,
                        "stay": stay_date, "date": comment_date, "content": content
                    })
                except:
                    continue

            if len(all_results) >= target_count: break

            # 4. 執行翻頁 (使用你截圖中的下一個評論頁面按鈕)
            print("➡️ 嘗試點擊下一頁按鈕...")
            try:
                next_btn_selector = 'button[data-element-name="review-paginator-next"]'
                next_btn = driver.find_element(By.CSS_SELECTOR, next_btn_selector)
                
                # 滾動到按鈕位置並強制點擊
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
                time.sleep(1)
                driver.execute_script("arguments[0].click();", next_btn)
                
                time.sleep(5) # 等待 AJAX 載入新內容
            except Exception as e:
                print(f"🛑 無法翻頁或已達末頁")
                break

        # 5. 最終終端機輸出
        print("\n" + "★" * 20 + f" 抓取完成：共 {len(all_results)} 則 " + "★" * 20)
        for i, res in enumerate(all_results):
            print(f"【第 {i+1} 則】 ID: {res['id']}")
            print(f"👤 用戶: {res['name']}  |  ⭐ 分數: {res['score']}")
            print(f"📅 發表: {res['date']}  |  🗓️ 入住: {res['stay']}")
            print(f"💬 內容: {res['content'][:150]}...")
            print("-" * 60)

    except Exception as e:
        print(f"❌ 執行出錯: {e}")
    finally:
        if 'driver' in locals():
            input("\n確認完畢請按 Enter 關閉瀏覽器...")
            driver.quit()

if __name__ == "__main__":
    run_agoda_spider(HOTEL_URL, MAX_REVIEWS)

📜 正在處理頁面，目前已成功抓取 0 則...
➡️ 嘗試點擊下一頁按鈕...
📜 正在處理頁面，目前已成功抓取 5 則...
➡️ 嘗試點擊下一頁按鈕...
📜 正在處理頁面，目前已成功抓取 10 則...

★★★★★★★★★★★★★★★★★★★★ 抓取完成：共 15 則 ★★★★★★★★★★★★★★★★★★★★
【第 1 則】 ID: 1070577683
👤 用戶: Adrian，來自菲律賓  |  ⭐ 分數: 10.0
📅 發表: 2026年1月入住 1 晚  |  🗓️ 入住: 2026年1月入住 1 晚
💬 內容: 我們選擇了台北圓環大飯店作為我們慶祝一周年婚禮的兩天住宿，這次體驗在各個方面都超出了我們的期望。從一開始，飯店就展現了對細節的卓越關注和真誠的款待。 我們對特別房間安排的請求得到了周到的滿足，並獲贈了一瓶免費的葡萄酒，這為我們的住宿增添了意義和慶祝的氛圍。這些舉動體現了飯店致力於定制客戶體驗的承諾，...
------------------------------------------------------------
【第 2 則】 ID: 1070579514
👤 用戶: Adrian，來自菲律賓  |  ⭐ 分數: 10.0
📅 發表: 2026年1月入住 1 晚  |  🗓️ 入住: 2026年1月入住 1 晚
💬 內容: 我們在台北圓環大飯店的兩天住宿體驗，透過 Agoda 預訂，實在是一次值得難忘的經歷，特別是因為這標誌著我們的第一次結婚周年紀念。飯店完美地平衡了歷史的宏偉與現代周到的服務。 工作人員真心努力滿足我們的特別要求，包括定制的客房安排和一瓶免費的葡萄酒。這些周到的細節深得我們的欣賞，為我們的入住增添了溫...
------------------------------------------------------------
【第 3 則】 ID: 1014205418
👤 用戶: CHIH-MIN，來自台灣  |  ⭐ 分數: 10.0
📅 發表: 2025年8月入住2晚  |  🗓️ 入住: 2025年8月入住2晚
💬 內容: 自然景觀和歷史建築風格令人驚豔。房務人員體貼細心，房間超級乾淨。在我們孩子的生日，他們甚至還送了精緻的小蛋糕，這樣的暖心舉動讓人感動。 特別值得一提的是服務中心的工作人員，他們在

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 設定目標
HOTEL_URL = "https://www.agoda.com/zh-hk/grand-hotel/hotel/taipei-tw.html"
MAX_REVIEWS = 15 

def run_agoda_spider(url, target_count):
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    all_results = []
    seen_ids = set()

    try:
        driver.get(url)
        wait = WebDriverWait(driver, 20)

        while len(all_results) < target_count:
            print(f"📜 正在處理頁面，目前已成功抓取 {len(all_results)} 則...")
            
            # 1. 深度滾動
            for _ in range(5):
                driver.execute_script("window.scrollBy(0, 1000);")
                time.sleep(0.5)
            
            # 2. 定位所有評論卡片
            card_selector = '[data-element-name="review-comment"]'
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, card_selector)))
            cards = driver.find_elements(By.CSS_SELECTOR, card_selector)

            for card in cards:
                if len(all_results) >= target_count: break
                
                try:
                    # 檢查 ID 防止重複
                    r_id = card.get_attribute("data-review-id")
                    if not r_id or r_id in seen_ids: continue
                    
                    # 排除沒有分數的（例如飯店回覆）
                    score_el = card.find_elements(By.CSS_SELECTOR, ".Review-comment-leftScore")
                    if not score_el: continue

                    # 【新增】處理「閱讀更多」按鈕，展開完整評論
                    try:
                        read_more_btn = card.find_elements(By.CSS_SELECTOR, '[data-element-name="review-read-more-button"]')
                        if read_more_btn:
                            driver.execute_script("arguments[0].click();", read_more_btn[0])
                            time.sleep(0.3) # 微等展開
                    except:
                        pass

                    # 提取基本欄位
                    score = score_el[0].text
                    name = card.find_element(By.CSS_SELECTOR, '[data-info-type="reviewer-name"]').text
                    
                    # 【修正 1】精確抓取「入住日期」
                    try:
                        stay_date = card.find_element(By.CSS_SELECTOR, '[data-info-type="stay-detail"]').text
                    except:
                        stay_date = "未知入住日期"

                    # 【修正 2】精確抓取「發表日期」 (避開入住日期)
                    comment_date = "未知發表日期"
                    # 根據截圖，日期通常在 Review-comment-right 裡面的特定 span
                    all_spans = card.find_elements(By.TAG_NAME, "span")
                    for s in all_spans:
                        txt = s.text
                        # 判斷是發表日期：包含 '202' 且 包含 '評價於' 或 '發表於'
                        if "202" in txt and ("評價" in txt or "發表" in txt):
                            comment_date = txt
                            break

                    # 【修正 3】抓取完整內容，不再手動截斷
                    content = card.find_element(By.CSS_SELECTOR, ".Review-comment-bodyText").text.strip()
                    try:
                        # 定位該評論卡片內所有的照片按鈕
                        # 使用 data-element-name 屬性是最穩定的做法
                        photo_buttons = card.find_elements(By.CSS_SELECTOR, 'button[data-element-name="review-comment-ugc-thumbnail"')
                        photo_count = len(photo_buttons)
                    except:
                        photo_count = 0
                    seen_ids.add(r_id)
                    all_results.append({
                        "id": r_id, "name": name, "score": score,
                        "stay": stay_date, "date": comment_date, "content": content,"photo_count": photo_count
                    })
                except Exception as e:
                    continue

            if len(all_results) >= target_count: break

            # 3. 執行翻頁
            print("➡️ 嘗試點擊下一頁按鈕...")
            try:
                next_btn_selector = 'button[data-element-name="review-paginator-next"]'
                next_btn = driver.find_element(By.CSS_SELECTOR, next_btn_selector)
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
                time.sleep(1)
                driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(5) 
            except:
                print(f"🛑 無法翻頁或已達末頁")
                break

        # 4. 最終輸出
        print("\n" + "★" * 20 + f" 抓取完成：共 {len(all_results)} 則 " + "★" * 20)
        for i, res in enumerate(all_results):
            print(f"【第 {i+1} 則】 ID: {res['id']}")
            print(f"👤 用戶: {res['name']}  |  ⭐ 分數: {res['score']} | 📷 照片數: {res['photo_count']}")
            print(f"📅 發表: {res['date']}")
            print(f"🗓️ 入住: {res['stay']}")
            # 【修正 4】這裡直接印出全部 content，不再加 [:150]
            print(f"💬 內容: {res['content']}")
            print("-" * 70)

    except Exception as e:
        print(f"❌ 執行出錯: {e}")
    finally:
        if 'driver' in locals():
            input("\n確認完畢請按 Enter 關閉瀏覽器...")
            driver.quit()

if __name__ == "__main__":
    run_agoda_spider(HOTEL_URL, MAX_REVIEWS)

📜 正在處理頁面，目前已成功抓取 0 則...
➡️ 嘗試點擊下一頁按鈕...
📜 正在處理頁面，目前已成功抓取 5 則...
➡️ 嘗試點擊下一頁按鈕...
📜 正在處理頁面，目前已成功抓取 10 則...

★★★★★★★★★★★★★★★★★★★★ 抓取完成：共 15 則 ★★★★★★★★★★★★★★★★★★★★
【第 1 則】 ID: 1085338828
👤 用戶: TAKAKO，來自日本  |  ⭐ 分數: 9.2 | 📷 照片數: 0
📅 發表: 2026年3月23日星期一評價
🗓️ 入住: 2026年3月入住 1 晚
💬 內容: 當日午前中到達，但他們樂意幫我保管行李；晚上再次辦理入住時，竟然將我升級到了景觀良好的房間。那是一間能一覽全市的美好房間。床也很大，廁所配有洗功能馬桶，浴缸也很寬敞，讓我可以好好放鬆。雖然距離車站有些距離，但提供免費的接駁巴士，所以並不覺得不方便。下次我想在酒店多待一些時間，慢慢享受。
----------------------------------------------------------------------
【第 2 則】 ID: 1088578081
👤 用戶: Laurie，來自美國  |  ⭐ 分數: 9.2 | 📷 照片數: 0
📅 發表: 2026年3月25日星期三評價
🗓️ 入住: 2026年3月入住3晚
💬 內容: 這家酒店值得入住，就為了親眼看看！我們訂了最便宜的內部房間，但仍然很棒！柔軟的地毯和毛巾、強力的淋浴等。清潔團隊非常棒，快速且在我們經過時也很友好。我們在圓圓用餐，食物不錯，但價格偏貴。非常友好的行李員，特別熱心地幫助我們。老公參加了隧道之旅，喜歡這個行程。唯一的缺點是距離一些景點有點遠，但接駁車真的很有幫助（他們總是準時）。我們使用捷運四處移動，但如果乘坐計程車的話，可能會很貴。他們也帶我們去了士林夜市，雖然我不喜歡那個市場，但還不錯。華麗的酒店，我會再回來的❤️
----------------------------------------------------------------------
【第 3 則】 ID: 1059506510
👤 用戶: HATSUMI，來自日本  |  ⭐ 分數: 9.6 | 📷 照片數: 0
📅 發表: 2025年12月16日星期二評價


In [ ]:
import time
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 設定目標
HOTEL_NAME = "台北圓山大飯店" 
HOTEL_URL = "https://www.agoda.com/zh-hk/grand-hotel/hotel/taipei-tw.html"
MAX_REVIEWS = 15 

def run_agoda_spider(url, target_count):
    options = webdriver.ChromeOptions()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    all_results = []
    seen_ids = set()

    try:
        driver.get(url)
        wait = WebDriverWait(driver, 20)

        while len(all_results) < target_count:
            print(f"📜 正在處理頁面，目前已成功抓取 {len(all_results)} 則...")
            
            # 深度滾動確保元素加載
            for _ in range(5):
                driver.execute_script("window.scrollBy(0, 1000);")
                time.sleep(0.5)
            
            card_selector = '[data-element-name="review-comment"]'
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, card_selector)))
            cards = driver.find_elements(By.CSS_SELECTOR, card_selector)

            for card in cards:
                if len(all_results) >= target_count: break
                
                try:
                    r_id = card.get_attribute("data-review-id")
                    if not r_id or r_id in seen_ids: continue
                    
                    score_el = card.find_elements(By.CSS_SELECTOR, ".Review-comment-leftScore")
                    if not score_el: continue

                    # 展開「閱讀更多」
                    try:
                        read_more_btn = card.find_elements(By.CSS_SELECTOR, '[data-element-name="review-read-more-button"]')
                        if read_more_btn:
                            driver.execute_script("arguments[0].click();", read_more_btn[0])
                            time.sleep(0.3)
                    except: pass

                    # 提取基本資料
                    score = score_el[0].text
                    content = card.find_element(By.CSS_SELECTOR, ".Review-comment-bodyText").text.strip()
                    
                    # 處理日期
                    comment_date = "未知日期"
                    all_spans = card.find_elements(By.TAG_NAME, "span")
                    for s in all_spans:
                        txt = s.text
                        if "202" in txt and ("評價" in txt or "發表" in txt):
                            comment_date = txt
                            break

                    # 嘗試提取「房型」與「旅遊類型」 (通常在評論者名稱下方的特定容器)
                    room_type = ""
                    travel_type = ""
                    try:
                        detail_info = card.find_elements(By.CSS_SELECTOR, '.Review-comment-reviewer-subInfo .Review-comment-reviewer-subInfo-item')
                        if len(detail_info) >= 1: travel_type = detail_info[0].text
                        if len(detail_info) >= 2: room_type = detail_info[1].text
                    except: pass

                    # 抓取照片數量
                    photo_count = 0
                    try:
                        photo_buttons = card.find_elements(By.CSS_SELECTOR, 'button[data-element-name="review-comment-ugc-thumbnail"]')
                        photo_count = len(photo_buttons)
                    except: pass

                    # 封裝成 JSON 格式
                    review_item = {
                        "飯店名稱": HOTEL_NAME,
                        "評論ID": r_id,
                        "評分": score,
                        "評論內容": content,
                        "評論日期": comment_date,
                        "旅遊類型": travel_type,
                        "房型": room_type,
                        "照片數量": photo_count
                    }

                    seen_ids.add(r_id)
                    all_results.append(review_item)

                    # --- 同步在終端輸出 ---
                    print(f"✅ 已抓取 ID: {r_id} | ⭐: {score} | 📷: {photo_count}")

                except Exception:
                    continue

            if len(all_results) >= target_count: break

            # 翻頁邏輯
            try:
                next_btn = driver.find_element(By.CSS_SELECTOR, 'button[data-element-name="review-paginator-next"]')
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
                time.sleep(1)
                driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(5) 
            except:
                print("🛑 無法翻頁，停止抓取")
                break

        # --- 導出 JSON 檔案 ---
        file_name = "agoda_reviews.json"
        with open(file_name, "w", encoding="utf-8") as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)
        
        print("\n" + "★" * 20 + " 最終終端報告 " + "★" * 20)
        for i, res in enumerate(all_results):
            print(f"【{i+1}】 {res['評論日期']} | {res['評分']}分 | 照片: {res['照片數量']}張")
            print(f"   內容: {res['評論內容'][:50]}...") # 終端機只印前50字避免刷屏

        print(f"\n📂 數據已成功存入: {file_name}")

    except Exception as e:
        print(f"❌ 執行出錯: {e}")
    finally:
        if 'driver' in locals():
            input("\n確認完畢請按 Enter 關閉瀏覽器...")
            driver.quit()

if __name__ == "__main__":
    run_agoda_spider(HOTEL_URL, MAX_REVIEWS)

📜 正在處理頁面，目前已成功抓取 0 則...
✅ 已抓取 ID: 1070577683 | ⭐: 10.0 | 📷: 2
✅ 已抓取 ID: 1070579514 | ⭐: 10.0 | 📷: 0
✅ 已抓取 ID: 1014205418 | ⭐: 10.0 | 📷: 0
✅ 已抓取 ID: 1064938054 | ⭐: 8.8 | 📷: 0
✅ 已抓取 ID: 1077342253 | ⭐: 10.0 | 📷: 1
📜 正在處理頁面，目前已成功抓取 5 則...
✅ 已抓取 ID: 977313694 | ⭐: 9.6 | 📷: 0
✅ 已抓取 ID: 1064935880 | ⭐: 8.4 | 📷: 0
✅ 已抓取 ID: 967950681 | ⭐: 9.6 | 📷: 0
✅ 已抓取 ID: 1050040636 | ⭐: 9.6 | 📷: 0
✅ 已抓取 ID: 902945491 | ⭐: 9.6 | 📷: 0
📜 正在處理頁面，目前已成功抓取 10 則...
✅ 已抓取 ID: 938839422 | ⭐: 10.0 | 📷: 0
✅ 已抓取 ID: 1063911456 | ⭐: 9.2 | 📷: 0
✅ 已抓取 ID: 1071416519 | ⭐: 9.6 | 📷: 0
✅ 已抓取 ID: 827531626 | ⭐: 8.4 | 📷: 0
✅ 已抓取 ID: 1078738974 | ⭐: 9.6 | 📷: 0

★★★★★★★★★★★★★★★★★★★★ 最終終端報告 ★★★★★★★★★★★★★★★★★★★★
【1】 2026年1月26日星期一評價 | 10.0分 | 照片: 2張
   內容: 我們選擇了台北圓環大飯店作為我們慶祝一周年婚禮的兩天住宿，這次體驗在各個方面都超出了我們的期望。從一...
【2】 2026年1月26日星期一評價 | 10.0分 | 照片: 0張
   內容: 我們在台北圓環大飯店的兩天住宿體驗，透過 Agoda 預訂，實在是一次值得難忘的經歷，特別是因為這標...
【3】 2025年8月26日星期二評價 | 10.0分 | 照片: 0張
   內容: 自然景觀和歷史建築風格令人驚豔。房務人員體貼細心，房間超級乾淨。在我們孩子的生日，他們甚至還送了精緻...
【4】 2026年1月7日星期三評價 | 8.8分 | 

In [ ]:
url="https://www.agoda.com/zh-hk/grand-hotel/reviews/taipei-tw.html"